### Baseline Model Development

Goal:
Build the first baseline model for loan risk prediction using the cleaned dataset.

#### Baseline Model Development

Goal:
Predict whether a borrower will default on a loan.

Target:
is_default
0 = Fully Paid
1 = Charged Off / Default

Current Dataset:
X.shape = (87889, 89)
y.shape = (87889,)

Planned Baseline:
Logistic Regression

Evaluation Strategy:
- Stratified Train/Test Split
- Accuracy not used as primary metric
- Focus on classification metrics suitable for imbalanced data

## Next Steps

- Perform stratified 80/20 train-test split
- Train Logistic Regression baseline
- Evaluate using ROC-AUC, Precision, Recall, F1-score
- Compare with future tree-based models

In [2]:
import pandas as pd

In [3]:
X = pd.read_csv("../data/processed/X.csv", index_col=0)

In [4]:
X.shape

(87889, 90)

In [5]:
y =pd.read_csv("../data/processed/y.csv", index_col=0).squeeze()

In [6]:
y.shape

(87889,)

In [7]:
X.index.equals(y.index)

True

In [8]:
from sklearn.model_selection import train_test_split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y, 
    test_size=0.2,
    random_state=42, 
    stratify=y
)

In [10]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (70311, 90)
X_test : (17578, 90)
y_train: (70311,)
y_test : (17578,)


In [20]:
print("Training distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True))

Training distribution:
loan_status
0    0.799704
1    0.200296
Name: proportion, dtype: float64

Test distribution:
loan_status
0    0.799693
1    0.200307
Name: proportion, dtype: float64


In [11]:
X.dtypes.value_counts()

float64    68
bool       14
int64       8
Name: count, dtype: int64

In [12]:
X.select_dtypes(include="bool").columns.tolist()


['MORTGAGE',
 'OWN',
 'RENT',
 'credit_card',
 'debt_consolidation',
 'home_improvement',
 'house',
 'major_purchase',
 'medical',
 'moving',
 'other',
 'renewable_energy',
 'small_business',
 'vacation']

In [13]:
X.select_dtypes(include="int64").columns.to_list()

['id',
 'term',
 'sub_grade',
 'verification_status',
 'initial_list_status',
 'application_type',
 'debt_settlement_flag',
 'is_default']

In [14]:
y.value_counts()

loan_status
0    70285
1    17604
Name: count, dtype: int64

In [15]:
X["is_default"].value_counts()

is_default
0    70285
1    17604
Name: count, dtype: int64

In [16]:
X[['term', 'sub_grade', 'verification_status',
   'initial_list_status', 'application_type',
   'debt_settlement_flag']].nunique()

term                     2
sub_grade               35
verification_status      3
initial_list_status      2
application_type         2
debt_settlement_flag     2
dtype: int64

In [17]:
X[['term', 'sub_grade', 'verification_status',
   'initial_list_status', 'application_type',
   'debt_settlement_flag']].head()

,term,sub_grade,verification_status,initial_list_status,application_type,debt_settlement_flag
0,0,13,0,0,1,0
1,0,10,0,0,1,0
2,1,8,0,0,0,0
4,1,25,1,0,1,0
5,0,12,1,0,1,0


In [18]:
X.select_dtypes(include="float64").columns.tolist()

['loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'int_rate',
 'installment',
 'emp_length',
 'annual_inc',
 'dti',
 'delinq_2yrs',
 'fico_range_low',
 'fico_range_high',
 'inq_last_6mths',
 'mths_since_last_delinq',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'out_prncp',
 'out_prncp_inv',
 'recoveries',
 'collection_recovery_fee',
 'last_fico_range_high',
 'last_fico_range_low',
 'collections_12_mths_ex_med',
 'policy_code',
 'acc_now_delinq',
 'tot_coll_amt',
 'tot_cur_bal',
 'total_rev_hi_lim',
 'acc_open_past_24mths',
 'avg_cur_bal',
 'bc_open_to_buy',
 'bc_util',
 'chargeoff_within_12_mths',
 'delinq_amnt',
 'mo_sin_old_il_acct',
 'mo_sin_old_rev_tl_op',
 'mo_sin_rcnt_rev_tl_op',
 'mo_sin_rcnt_tl',
 'mort_acc',
 'mths_since_recent_bc',
 'mths_since_recent_inq',
 'mths_since_recent_revol_delinq',
 'num_accts_ever_120_pd',
 'num_actv_bc_tl',
 'num_actv_rev_tl',
 'num_bc_sats',
 'num_bc_tl',
 'num_il_tl',
 'num_op_rev_tl',
 'num_rev_accts',
 'num_rev_tl_bal

In [19]:
X.select_dtypes(include="float64").nunique().sort_values().to_frame("n_unique")

,n_unique
policy_code,1
out_prncp_inv,2
out_prncp,2
num_tl_120dpd_2m,3
num_tl_30dpd,5
...,...
revol_bal,35372
total_il_high_credit_limit,50710
total_bal_ex_mort,59777
tot_hi_cred_lim,70343


In [20]:
X.select_dtypes(include="float64").nunique().sort_values().head(15)

policy_code                    1
out_prncp_inv                  2
out_prncp                      2
num_tl_120dpd_2m               3
num_tl_30dpd                   5
acc_now_delinq                 5
inq_last_6mths                 6
collections_12_mths_ex_med     6
pub_rec_bankruptcies           8
chargeoff_within_12_mths       8
emp_length                    12
num_tl_90g_dpd_24m            17
delinq_2yrs                   19
tax_liens                     20
pub_rec                       22
dtype: int64

In [21]:
X.select_dtypes(include="float64").nunique().sort_values().head(25)

policy_code                    1
out_prncp_inv                  2
out_prncp                      2
num_tl_120dpd_2m               3
num_tl_30dpd                   5
acc_now_delinq                 5
inq_last_6mths                 6
collections_12_mths_ex_med     6
pub_rec_bankruptcies           8
chargeoff_within_12_mths       8
emp_length                    12
num_tl_90g_dpd_24m            17
delinq_2yrs                   19
tax_liens                     20
pub_rec                       22
mort_acc                      24
mths_since_recent_inq         26
num_tl_op_past_12m            26
num_actv_bc_tl                27
num_accts_ever_120_pd         29
num_rev_tl_bal_gt_0           37
fico_range_low                38
fico_range_high               38
num_bc_sats                   38
num_actv_rev_tl               40
dtype: int64

In [22]:
X_train.dtypes.value_counts()

float64    68
bool       14
int64       8
Name: count, dtype: int64

In [24]:
excluded_cols = [
    "id",
    "is_default",
    "debt_settlement_flag",
    "policy_code",
    "out_prncp",
    "out_prncp_inv",
]

In [25]:
[col for col in [
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "next_pymnt_d",
] if col in X.columns]

[]

In [26]:
X_train = X_train.drop(columns=excluded_cols)
X_test = X_test.drop(columns=excluded_cols)

In [27]:
[col for col in excluded_cols if col in X_train.columns or col in X_test.columns]

[]

In [28]:
numerical_cols = X_train.select_dtypes(include=["int64","float64"]).columns.tolist()
numerical_cols

['loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'sub_grade',
 'emp_length',
 'annual_inc',
 'verification_status',
 'dti',
 'delinq_2yrs',
 'fico_range_low',
 'fico_range_high',
 'inq_last_6mths',
 'mths_since_last_delinq',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'recoveries',
 'collection_recovery_fee',
 'last_fico_range_high',
 'last_fico_range_low',
 'collections_12_mths_ex_med',
 'application_type',
 'acc_now_delinq',
 'tot_coll_amt',
 'tot_cur_bal',
 'total_rev_hi_lim',
 'acc_open_past_24mths',
 'avg_cur_bal',
 'bc_open_to_buy',
 'bc_util',
 'chargeoff_within_12_mths',
 'delinq_amnt',
 'mo_sin_old_il_acct',
 'mo_sin_old_rev_tl_op',
 'mo_sin_rcnt_rev_tl_op',
 'mo_sin_rcnt_tl',
 'mort_acc',
 'mths_since_recent_bc',
 'mths_since_recent_inq',
 'mths_since_recent_revol_delinq',
 'num_accts_ever_120_pd',
 'num_actv_bc_tl',
 'num_actv_rev_tl',
 'num_bc_sats',
 'num_bc_tl',
 'num_il_tl',
 'num_op

In [32]:
binary_cols = X_train.select_dtypes(include=["bool"]).columns.to_list()
binary_cols

['MORTGAGE',
 'OWN',
 'RENT',
 'credit_card',
 'debt_consolidation',
 'home_improvement',
 'house',
 'major_purchase',
 'medical',
 'moving',
 'other',
 'renewable_energy',
 'small_business',
 'vacation']

In [33]:
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.to_list()
categorical_cols

[]

In [34]:
categorical_cols = ["verification_status"]

In [35]:
"verification_status" in numerical_cols

True

In [36]:
numerical_cols.remove("verification_status")

In [37]:
"verification_status" in numerical_cols

False

In [38]:
low_cardinality = (
    X_train.nunique()
    .sort_values()
    .loc[lambda s: (s >= 3) & (s <= 12)]
)

low_cardinality

verification_status            3
num_tl_120dpd_2m               3
acc_now_delinq                 4
num_tl_30dpd                   4
collections_12_mths_ex_med     6
inq_last_6mths                 6
pub_rec_bankruptcies           7
chargeoff_within_12_mths       8
emp_length                    12
dtype: int64